In [2]:
from mlflow.tracking import MlflowClient

In [3]:
DB_PATH = "/workspaces/mlops-zoomcamp/02-experiment-tracking/mlflow.db"

MLFLOW_TRACKING_URI = f"sqlite:///{DB_PATH}"

In [ ]:
# mlflow ui --backend-store-uri sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/mlflow.db

In [4]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [5]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1776672140948, experiment_id='2', last_update_time=1776672140948, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1776061263776, experiment_id='1', last_update_time=1776061263776, lifecycle_stage='active', name='duration-prediction', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1776061263770, experiment_id='0', last_update_time=1776061263770, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [6]:
# client.create_experiment(name="my-cool-experiment")

In [7]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids="1",
    filter_string="metrics.rmse <6.8 ",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]

)

for run in runs:
    print(f"run_id: {run.info.run_id}, rmse: {run.data.metrics['rmse']: 4f}")

run_id: d6a6e3a4688c4080ae99754d47d525cc, rmse:  6.318446
run_id: 15ce7961a07148a9bf74db44a6405060, rmse:  6.391028
run_id: 9924c0db8b724238b1f7fd76963ba905, rmse:  6.417410
run_id: 650cf1166a1c45dcb7bdfade8143fc41, rmse:  6.478640
run_id: 59c9635e7659471cade01e548ad39dfe, rmse:  6.479841
run_id: bd3400c9c7174de08d8d85e3bc4fc0ef, rmse:  6.593269
run_id: 61efa9ca47e84c87acf975b0a9b1e286, rmse:  6.600157


In [8]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [11]:
run_id = "c34e8acacfe647a2a82320a6687d6872"
model_uri_to_register = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri_to_register, name="nyc-taxi-regressor")


Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/04/20 08:12:53 WARNING mlflow.tracking._model_registry.fluent: Run with id c34e8acacfe647a2a82320a6687d6872 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-f5ee30b061284ed098a56d94be7870d9 instead


Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1776672773453, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [13]:
# Trasition model from one stage to another
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1776669987288, deployment_job_id=None, deployment_job_state=None, description='NYC Taxi duration prediction', last_updated_timestamp=1776672773453, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1776670068425, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1776670729845, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='d5efa482a4c94f5a976a3f45b62105ff', run_link='', source='models:/m-cdc5477db7844f37aa3caf0cca141c55', status='READY', status_message=None, tags={'estimator name': 'linear regression'}, user_id=None, version=2, workspace='default'>,
  <ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1776672773453, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:

In [26]:
latests_versions = client.get_latest_versions(name="nyc-taxi-regressor") # returns latest version for each stage

for version in latests_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Production
version: 2, stage: Staging
version: 4, stage: None


/tmp/ipykernel_5776/2245416359.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latests_versions = client.get_latest_versions(name="nyc-taxi-regressor") # returns latest version for each stage


In [29]:
version = 4
stage = "Staging"
client.transition_model_version_stage(name="nyc-taxi-regressor",
                                      version=version,
                                      stage=stage,
                                      archive_existing_versions=False)


/tmp/ipykernel_5776/3195850804.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name="nyc-taxi-regressor",


<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1776673552234, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [31]:
import datetime
date = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
client.update_model_version(name="nyc-taxi-regressor",
                            version=version,
                            description=f"This model version ({version}) was transitioned to {stage} on {date}"
                            )

<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='Staging', deployment_job_state=None, description='This model version (4) was transitioned to Staging on 2026-04-20 08:26:00', last_updated_timestamp=1776673560653, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [ ]:
# Lets say we want to move the current prod model to archived and promote a staged model to prod